# Fine-tuning Qwen2.5-0.5B-Instruct for Customer Support Conversations

This notebook fine-tunes a small instruction model using LoRA on cleaned conversation data.

In [2]:
# Install dependencies
!pip install -q transformers peft accelerate datasets torch scikit-learn

In [3]:
from google.colab import files
files.upload()

Saving cleaned_conversations.jsonl to cleaned_conversations.jsonl


{'cleaned_conversations.jsonl': b'{"conversation_id": "conv_001", "language": "hindi", "turns": [{"role": "agent", "text": "\xe0\xa4\xa8\xe0\xa4\xae\xe0\xa4\xb8\xe0\xa5\x8d\xe0\xa4\x95\xe0\xa4\xbe\xe0\xa4\xb0, EMI \xe0\xa4\x95\xe0\xa4\xb2\xe0\xa5\x87\xe0\xa4\x95\xe0\xa5\x8d\xe0\xa4\xb6\xe0\xa4\xa8 \xe0\xa4\xb8\xe0\xa5\x87 \xe0\xa4\x95\xe0\xa5\x89\xe0\xa4\xb2 \xe0\xa4\x95\xe0\xa4\xbf\xe0\xa4\xaf\xe0\xa4\xbe \xe0\xa4\xb9\xe0\xa5\x88\xe0\xa5\xa4 \xe0\xa4\x95\xe0\xa5\x8d\xe0\xa4\xaf\xe0\xa4\xbe \xe0\xa4\xae\xe0\xa5\x88\xe0\xa4\x82 \xe0\xa4\x86\xe0\xa4\xaa\xe0\xa4\x95\xe0\xa5\x80 \xe0\xa4\xb8\xe0\xa4\xb9\xe0\xa4\xbe\xe0\xa4\xaf\xe0\xa4\xa4\xe0\xa4\xbe \xe0\xa4\x95\xe0\xa4\xb0 \xe0\xa4\xb8\xe0\xa4\x95\xe0\xa4\xa4\xe0\xa4\xbe \xe0\xa4\xb9\xe0\xa5\x82\xe0\xa4\x81?"}, {"role": "customer", "text": "\xe0\xa4\xb8\xe0\xa4\xb0, \xe0\xa4\x95\xe0\xa5\x83\xe0\xa4\xaa\xe0\xa4\xaf\xe0\xa4\xbe EMI \xe0\xa4\xad\xe0\xa5\x81\xe0\xa4\x97\xe0\xa4\xa4\xe0\xa4\xbe\xe0\xa4\xa8 \xe0\xa4\x9c\xe0\xa4\xb2\xe0\xa5\x8d

In [4]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset


In [5]:
# If using Google Drive, uncomment and mount:
# from google.colab import drive
# drive.mount('/content/drive')
# data_file = '/content/drive/My Drive/cleaned_conversations.jsonl'

# Otherwise, use the local path (relative to this notebook)
data_file = '/content/cleaned_conversations.jsonl'

data = []
with open(data_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} conversations")


Loaded 90 conversations


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Prepare dataset for instruction tuning
instructions = []
responses = []
for conv in data:
    turns = conv['turns']
    for i in range(0, len(turns)-1, 2):
        if i+1 < len(turns):
            instructions.append(turns[i]['text'])
            responses.append(turns[i+1]['text'])

# Limit to 50 examples (fast training on Colab)
instructions = instructions[:50]
responses = responses[:50]

print(f"Prepared {len(instructions)} instruction-response pairs")


Prepared 50 instruction-response pairs


In [8]:
# Load model and tokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
# Set up LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [10]:
# Tokenize and split
def tokenize_function(examples):
    texts = [f"Instruction: {instr}\nResponse: {resp}" for instr, resp in zip(examples['instruction'], examples['response'])]
    tokenized = tokenizer(texts, truncation=True, padding=True, max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()  # For causal LM loss computation
    return tokenized

dataset = Dataset.from_dict({'instruction': instructions, 'response': responses})
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Use Hugging Face dataset split (avoids sklearn issues)
split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Train size: 40, Eval size: 10


In [11]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,10.247087,11.520693
2,8.026582,9.676395
3,7.477624,8.881350


TrainOutput(global_step=60, training_loss=8.973636245727539, metrics={'train_runtime': 17.5953, 'train_samples_per_second': 6.82, 'train_steps_per_second': 3.41, 'total_flos': 27356047994880.0, 'train_loss': 8.973636245727539, 'epoch': 3.0})

In [12]:
# Save weights and run a quick evaluation
# Save weights
model.save_pretrained("./results")

# Evaluation
results = trainer.evaluate()
print(results)

# Sample inference
input_text = "Instruction: Hello, I have an issue with my payment.\nResponse:"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_length=100)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Sample output:", response)


{'eval_loss': 8.881349563598633, 'eval_runtime': 0.5513, 'eval_samples_per_second': 18.138, 'eval_steps_per_second': 9.069, 'epoch': 3.0}
Sample output: Instruction: Hello, I have an issue with my payment.
Response: Hello! I'm here to help. Can you please provide me with more information about your payment? What is the status of it and what are the reasons for it? Additionally, can you tell me if there are any issues or problems that need attention? With this information, I will be able to better assist you in resolving your payment issue. Is there anything specific you would like me to do next? Let's get started on
